In [2]:
!pip install langchain_groq

## Prompt Template

In [3]:
from langchain_core.prompts import PromptTemplate


template = """Tell me a joke about {topic}. make sure the joke the {topic} in only"""


prompt = PromptTemplate(
    input_variables=["topic"],
    template=template,
)

In [4]:
prompt.invoke({'topic': 'animal'})

StringPromptValue(text='Tell me a joke about animal. make sure the joke the animal in only')

In [5]:
prompt.format(topic="cricket")

'Tell me a joke about cricket. make sure the joke the cricket in only'

In [6]:
# !pip install langchain_groq

In [7]:
import os
from langchain_groq import ChatGroq


from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# To create llm object, make sure you have 'GROQ_API_KEY' in your environment variable
# comment above line of code where we are fetching colab userdata.

llm = ChatGroq(temperature=0.9, model_name="qwen/qwen3.8-27b")
response = llm.invoke("hi")
response.content

'Hi! How can I help you today?'

In [8]:
input_content = prompt.invoke({'topic': 'cricket'})
print(llm.invoke(input_content).content)

What do you call a cricket with no legs?

A little stick. 🦗


In [9]:
llm.invoke(input_content)

AIMessage(content='It seems there is a small contradiction in your request: you asked for a joke where the "cricket" is **only** (in the text), but a cricket joke usually involves the insect or the sport.\n\nHowever, if you meant a joke where the word **"only"** is the punchline or central element, here is a clean one:\n\n**Why was the cricket so quiet?**  \nBecause it was **only** listening. 🦗\n\nIf you meant something else (like a joke *only* about the sport, or a very short joke), let me know!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 27, 'total_tokens': 147, 'completion_time': 0.23521533, 'completion_tokens_details': None, 'prompt_time': 0.002084048, 'prompt_tokens_details': None, 'queue_time': 0.004382913, 'total_time': 0.237299378}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_c7e30c203c', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0418

In [10]:
chain = prompt | llm

print(chain.invoke({"topic": 'cricket'}).content)

Why are cricket players bad at keeping secrets?

Because they always **bowls** out!


## Chat Prompt Template

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

prompt = prompt_template.invoke({"topic": "kids"})

response = llm.invoke(prompt)

In [13]:
print(response.content)

Here is a classic one for you:

**Why did the baby refuse to eat the peas?**

Because they were too **p.e.a.s.y**! 🟢😄


## Parser

In [14]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class SentimentParser(BaseModel):
    joke: str = Field(description="one liner joke for the user based on topic.")
    rating: str = Field(description = "return a number between 0 to 5, which is rating of joke as to how funny it was according to you on a scale of 0 to 5, 5 being highest")

# creating llm object with structured output parser
llm_parser = llm.with_structured_output(SentimentParser)

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

response = llm_parser.invoke(prompt_template.format_prompt(topic="kids"))

In [15]:
response

SentimentParser(joke="Why don't kids like math? Because it gives them problems.", rating='3')

In [16]:
response.joke

"Why don't kids like math? Because it gives them problems."

In [17]:
response.rating

'3'

In [18]:
!pip install langfuse==4.14.5

In [30]:
!pip show langfuse

Name: langfuse
Version: 4.14.5
Summary: Langfuse Python SDK - LLM observability/tracing, datasets, experiments, LLM-as-a-judge evaluation, and prompt management
Home-page: 
Author: langfuse
Author-email: langfuse <developers@langfuse.com>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: backoff, httpx, opentelemetry-api, opentelemetry-exporter-otlp-proto-http, opentelemetry-sdk, packaging, pydantic, wrapt
Required-by: 


In [25]:
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler
from langchain_core.prompts import ChatPromptTemplate

# Initialize Langfuse client with constructor arguments
Langfuse(
    public_key=userdata.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=userdata.get("LANGFUSE_SECRET_KEY"),
    host="https://cloud.langfuse.com"  # Optional: defaults to https://cloud.langfuse.com
)

# Get the configured client instance
langfuse = get_client()

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler()

# Run your chain with Langfuse tracing
response = llm_parser.invoke(prompt, config={"callbacks": [langfuse_handler]})

# # Flush events to Langfuse in short-lived applications
# langfuse.flush()

In [27]:
response.joke

'Why did the kid bring a ladder to school? Because he heard the education was high!'

In [29]:
response.rating

'3'